# ⚡ Swizzle Enterprise GPU Worker — Colab Pro High-RAM Node (A100 / L4)
This notebook turns your **Google Colab Pro GPU** into an authentic cloud worker powering **`https://swizzle-ten.vercel.app/`**.

It runs **vLLM** with **distributed prefix caching** and **PagedAttention**, exposing a secure HTTPS endpoint via **Cloudflare Quick Tunnels**.
Once the tunnel starts, paste the generated URL into your Vercel project's `SWIZZLE_UPSTREAM_URL` environment variable.

In [ ]:
# 1. Verify Colab Pro GPU Allocation (Expect NVIDIA A100-SXM4-40GB/80GB or L4)
!nvidia-smi

In [ ]:
# 2. Install vLLM and Cloudflared Tunnel Binary
!pip install -q vllm
!wget -q -nc https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
!dpkg -i cloudflared-linux-amd64.deb > /dev/null 2>&1
print("✅ vLLM and cloudflared installed successfully!")

In [ ]:
# 3. Start vLLM Model Server with Prefix Caching
import subprocess, time, socket, os, sys

MODEL_ID = "Qwen/Qwen2.5-Coder-7B-Instruct"  #@param {type:"string"}
MAX_MODEL_LEN = "8192"  #@param ["4096", "8192", "16384", "32768"]
GPU_MEM_UTIL = "0.90"   #@param ["0.85", "0.90", "0.92", "0.95"]
ENABLE_SPECULATIVE_PLD = False  #@param {type:"boolean"}

print(f"🚀 Spawning vLLM worker on port 8000 with model {MODEL_ID} (Context: {MAX_MODEL_LEN}, GPU Util: {GPU_MEM_UTIL})...")
vllm_cmd = [
    "python3", "-m", "vllm.entrypoints.openai.api_server",
    "--model", MODEL_ID,
    "--enable-prefix-caching",
    "--max-model-len", str(MAX_MODEL_LEN),
    "--gpu-memory-utilization", str(GPU_MEM_UTIL),
    "--trust-remote-code",
    "--port", "8000"
]

if ENABLE_SPECULATIVE_PLD:
    vllm_cmd.extend([
        "--spec-method", "ngram",
        "--spec-tokens", "5"
    ])

vllm_log = open("vllm.log", "w", buffering=1)
vllm_process = subprocess.Popen(vllm_cmd, stdout=vllm_log, stderr=subprocess.STDOUT)

ready = False
print("⏳ Downloading weights & compiling CUDA graphs (takes 1-3 minutes)...")
for attempt in range(180):
    ret = vllm_process.poll()
    if ret is not None:
        vllm_log.flush()
        print(f"\n❌ vLLM process exited unexpectedly with return code {ret}!")
        with open("vllm.log", "r") as f:
            lines = f.readlines()
            print("".join(lines[-40:]))
        raise RuntimeError("vLLM failed to start. See logs above.")

    try:
        s = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
        s.settimeout(1)
        s.connect(("127.0.0.1", 8000))
        s.close()
        ready = True
        print("\n\n✅ vLLM OpenAI API Server is READY on port 8000!")
        break
    except Exception:
        time.sleep(2)
        if attempt % 5 == 0:
            print(".", end="", flush=True)

if not ready:
    print("\n⚠️ Timed out waiting for port 8000.")
    with open("vllm.log", "r") as f:
        lines = f.readlines()
        print("".join(lines[-40:]))
    raise TimeoutError("vLLM server startup timed out.")

In [ ]:
# 4. Launch Cloudflare Tunnel and Connect to https://swizzle-ten.vercel.app
import re, time, urllib.request, subprocess
from IPython.display import display, HTML

# Anti-sleep background audio heartbeat (keeps Colab tab active)
display(HTML('''
    <audio id="silent-heartbeat" loop autoplay>
        <source src="data:audio/wav;base64,UklGRiQAAABXQVZFZm10IBAAAAABAAEAQB8AAEAfAAABAAgAZGF0YQAAAAA=" type="audio/wav">
    </audio>
    <script>console.log("⚡ Swizzle Anti-Sleep Heartbeat active.");</script>
'''))

def start_tunnel():
    print("🔗 Spawning Cloudflare Quick Tunnel (*.trycloudflare.com)...")
    tunnel_cmd = ["cloudflared", "tunnel", "--url", "http://localhost:8000"]
    p = subprocess.Popen(tunnel_cmd, stderr=subprocess.PIPE, universal_newlines=True)
    public_url = None
    for line in p.stderr:
        match = re.search(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", line)
        if match:
            public_url = match.group(0)
            print("\n" + "="*75)
            print("🚀 SWIZZLE COLAB PRO GPU WORKER ONLINE!")
            print(f"Tunnel Endpoint: {public_url}")
            print("-"*75)
            print("👉 HOW TO CONNECT TO YOUR LIVE VERCEL WEBSITE:")
            print("1. Go to your Vercel Dashboard -> Project 'swizzle-ten' -> Settings -> Environment Variables")
            print(f"2. Add or update: SWIZZLE_UPSTREAM_URL = {public_url}")
            print("3. Now https://swizzle-ten.vercel.app/v1/chat/completions routes directly to this GPU!")
            print(f"   (Or pass header 'X-Swizzle-Upstream: {public_url}' in any API client)")
            print("="*75 + "\n")
            break
    return p, public_url

tunnel_process, public_url = start_tunnel()


In [ ]:
# 5. Live Sanity Check & Continuous Watchdog
import json

# Test direct inference
try:
    test_payload = json.dumps({
        "model": MODEL_ID,
        "messages": [{"role": "user", "content": "Write a hello world in Python."}],
        "max_tokens": 32,
        "temperature": 0.2,
    }).encode("utf-8")
    req = urllib.request.Request(
        "http://127.0.0.1:8000/v1/chat/completions",
        data=test_payload,
        headers={"Content-Type": "application/json"},
    )
    t0 = time.time()
    with urllib.request.urlopen(req, timeout=10) as resp:
        res_data = json.loads(resp.read().decode("utf-8"))
    elapsed_ms = (time.time() - t0) * 1000.0
    content = res_data.get("choices", [{}])[0].get("message", {}).get("content", "")
    print(f"✅ Sanity test passed in {elapsed_ms:.1f}ms! Response excerpt:")
    print(content[:120].strip() + "...\n")
except Exception as e:
    print(f"⚠️ Sanity check warning: {e}")

print("🟢 Starting Watchdog to keep Colab worker active...")
uptime_ticks = 0
while True:
    time.sleep(30)
    uptime_ticks += 1
    try:
        urllib.request.urlopen("http://127.0.0.1:8000/v1/models", timeout=5)
        if uptime_ticks % 10 == 0:
            print(f"[Watchdog] 🟢 Worker healthy | Uptime: {uptime_ticks*30}s | Tunnel: {public_url}")
    except Exception as e:
        print(f"[Watchdog] ⚠️ Ping warning: {e}")

    if tunnel_process.poll() is not None:
        print("[Watchdog] 🔄 Cloudflare tunnel dropped. Auto-recovering...")
        tunnel_process, public_url = start_tunnel()
